# 01. Clean
Tidies the raw DAIR crosstabs and IPEDS files into the CSVs in `data/clean/`. Blank-cell counts and sanity checks are printed so nothing is dropped silently.

In [1]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW, CLEAN = ROOT / "data/raw", ROOT / "data/clean"
UNZ = RAW / "ipeds/unz"
CLEAN.mkdir(exist_ok=True)

SHORT = {
    "Agricultural, Consumer, & Environmental Sciences (ACES)": "ACES",
    "Applied Health Sciences (AHS)": "AHS", "College of Media": "Media",
    "Division of Explatory Studies (DES)": "DES",  # DAIR's spelling, kept so the key matches the source
    "Education": "Education", "Fine & Applied Arts (FAA)": "FAA",
    "Gies College of Business": "Gies", "Grainger College of Engineering": "Grainger",
    "Liberal Arts & Sciences (LAS)": "LAS", "School of Information Sciences": "iSchool",
    "School of Social Work (SSW)": "SSW",
}
def num(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "").str.strip(), errors="coerce")

## DAIR retention
First-fall to second-fall retention by college at entry. Cohort size and number retained are kept so rates can be recomputed.

In [2]:
ret = pd.read_csv(RAW / "dair/dair_retention_by_college.csv", sep="\t", encoding="utf-16", header=1)
ret.columns = ["college", "measure"] + [c.split()[0] for c in ret.columns[2:]]
ret["measure"] = ret["measure"].str.strip()
ret = ret.melt(["college", "measure"], var_name="cohort", value_name="value")
print("retention blank cells:", ret["value"].isna().sum())
ret["value"] = num(ret["value"].astype(str).str.rstrip("%"))
ret = ret.pivot_table(index=["college", "cohort"], columns="measure", values="value").reset_index()
retention = pd.DataFrame({"college": ret["college"], "cohort": ret["cohort"].astype(int),
    "metric": "First-year retention", "cohort_size": ret["Size of Adjusted Cohort"],
    "count": ret["Number Retained"]})

retention blank cells: 0


## DAIR graduation
The non-cumulative file reports graduates in each year group. Cumulative rates are built from it: 4-year = groups 3 + 4, 5-year adds group 5, 6-year adds group 6. A blank group is treated as zero graduates, and the check below proves that against DAIR's own cumulative file. Each group has a stray fourth row mislabeled "Size of Adjusted Cohort" (no trailing spaces) that repeats the graduate count, so it is dropped.

In [3]:
grad = pd.read_csv(RAW / "dair/dair_graduation_noncumulative_by_college.csv", sep="\t", encoding="utf-16")
grad.columns = ["college", "group", "raw_measure"] + [c.split()[0] for c in grad.columns[3:]]
grad = grad[grad["raw_measure"].isin(["Size of Adjusted Cohort  ", "Graduate Count "])].copy()
grad["measure"] = grad["raw_measure"].str.strip()
grad = grad.drop(columns="raw_measure").melt(["college", "group", "measure"], var_name="cohort", value_name="value")
blank_groups = grad[(grad["measure"] == "Graduate Count") & grad["value"].isna()]
print("blank graduate-count cells (treated as 0):", len(blank_groups))
print(blank_groups[["college", "group", "cohort"]].to_string(index=False))
grad["value"] = num(grad["value"])
grads = grad[grad["measure"] == "Graduate Count"].fillna({"value": 0}).pivot_table(
    index=["college", "cohort"], columns="group", values="value", aggfunc="sum").fillna(0)

official = pd.read_csv(RAW / "dair/dair_graduation_6yr_by_college.csv", sep="\t", encoding="utf-16", header=1)
official.columns = ["college", "measure"] + [c.split()[0] for c in official.columns[2:]]
official["measure"] = official["measure"].str.strip()
official = official.melt(["college", "measure"], var_name="cohort", value_name="value")
official["value"] = num(official["value"].astype(str).str.rstrip("%"))
official = official.pivot_table(index=["college", "cohort"], columns="measure", values="value")
sizes = official["Size of Adjusted Cohort"]

# The School of Information Sciences has no entering cohort before 2020, so those rows do not exist.
no_cohort = grads.index.difference(official.index)
print("college x cohort rows with no cohort, dropped:", list(no_cohort))
assert all(college == "School of Information Sciences" for college, _ in no_cohort)
grads = grads.drop(no_cohort)
six = grads.sum(axis=1)
assert (six == official["Graduate Count"].reindex(six.index)).all(), "year groups do not sum to DAIR's cumulative count"
print("year groups sum to DAIR's cumulative graduate count for all", len(six), "college x cohort rows")

parts = []
for metric, groups in {"4-year graduation": ["3-Year", "4-Year"],
                       "5-year graduation": ["3-Year", "4-Year", "5-Year"],
                       "6-year graduation": ["3-Year", "4-Year", "5-Year", "6-Year"]}.items():
    part = grads[groups].sum(axis=1).rename("count").reset_index()
    part["metric"] = metric
    part["cohort"] = part["cohort"].astype(int)
    parts.append(part)
graduation = pd.concat(parts).merge(
    sizes.rename("cohort_size").reset_index().assign(cohort=lambda d: d["cohort"].astype(int)), on=["college", "cohort"])

blank graduate-count cells (treated as 0): 24
                       college  group cohort
School of Information Sciences 5-Year   2016
School of Information Sciences 4-Year   2016
School of Information Sciences 3-Year   2016
   School of Social Work (SSW) 6-Year   2016
              College of Media 6-Year   2017
                     Education 3-Year   2017
      Gies College of Business 6-Year   2017
School of Information Sciences 5-Year   2017
School of Information Sciences 4-Year   2017
School of Information Sciences 3-Year   2017
   School of Social Work (SSW) 6-Year   2017
              College of Media 6-Year   2018
                     Education 6-Year   2018
School of Information Sciences 5-Year   2018
School of Information Sciences 4-Year   2018
School of Information Sciences 3-Year   2018
   School of Social Work (SSW) 3-Year   2018
                     Education 6-Year   2019
School of Information Sciences 5-Year   2019
School of Information Sciences 4-Year   2019
School of

## Combine and add the campus row
DAIR's download has no campus row, so "All campus" is the sum of the 11 entering units. The retention total is checked against the campus figures on DAIR's dashboard (2021: 8,273 / 7,720; 2025: 9,195 / 8,724).

In [4]:
rates = pd.concat([retention, graduation], ignore_index=True)
rates["college_short"] = rates["college"].map(SHORT)
assert rates["college_short"].notna().all()
campus = rates.groupby(["cohort", "metric"], as_index=False)[["cohort_size", "count"]].sum()
campus["college"] = campus["college_short"] = "All campus"
rates = pd.concat([rates, campus], ignore_index=True)
rates["rate_pct"] = (100 * rates["count"] / rates["cohort_size"]).round(2)
rates["cohort_size"] = rates["cohort_size"].astype(int)
rates["count"] = rates["count"].astype(int)
rates = rates[["college", "college_short", "cohort", "metric", "cohort_size", "count", "rate_pct"]].sort_values(
    ["metric", "college_short", "cohort"])

check = rates[(rates["college"] == "All campus") & (rates["metric"] == "First-year retention")].set_index("cohort")
dashboard_totals = {2021: (8273, 7720), 2022: (7943, 7492), 2023: (8314, 7882), 2024: (9002, 8582), 2025: (9195, 8724)}
for cohort, (size, kept) in dashboard_totals.items():
    print(cohort, "sum of colleges", check.loc[cohort, "cohort_size"], check.loc[cohort, "count"], "| DAIR campus", size, kept)
    assert abs(check.loc[cohort, "cohort_size"] - size) <= 0.01 * size

assert rates["rate_pct"].between(0, 100).all()
assert not rates.duplicated(["college", "cohort", "metric"]).any()
assert rates.loc[rates["metric"] == "First-year retention", "cohort"].between(2021, 2025).all()
assert rates.loc[rates["metric"] != "First-year retention", "cohort"].between(2016, 2020).all()
rates.to_csv(CLEAN / "dair_rates.csv", index=False)
rates.shape

2021 sum of colleges 8273 7720 | DAIR campus 8273 7720
2022 sum of colleges 7943 7492 | DAIR campus 7943 7492
2023 sum of colleges 8314 7882 | DAIR campus 8314 7882
2024 sum of colleges 9002 8582 | DAIR campus 9002 8582
2025 sum of colleges 9195 8724 | DAIR campus 9195 8724


(228, 7)

## DAIR enrollment
Fall headcount by college and level, from the Historical Table crosstab. Skipped with a message if the raw file is not present yet.

In [5]:
enroll_path = RAW / "dair/dair_enrollment_historical.csv"
if enroll_path.exists():
    enr = pd.read_csv(enroll_path, sep="\t", encoding="utf-16", header=None, dtype=str)
    years = enr.iloc[0, 2:].str.extract(r"(\d{4})")[0]
    header_row = 0
    if years.notna().sum() < 5:
        header_row = 1
        years = enr.iloc[1, 2:].str.extract(r"(\d{4})")[0]
    body = enr.iloc[header_row + 1:].copy()
    body.columns = ["college_raw", "level"] + years.tolist()
    body["college_raw"] = body["college_raw"].ffill()
    long = body.melt(["college_raw", "level"], var_name="fall_year", value_name="headcount").dropna(subset=["fall_year"])
    print("blank enrollment cells:", long["headcount"].isna().sum(), "of", len(long))
    long["headcount"] = num(long["headcount"])
    long = long.dropna(subset=["headcount"])
    long["fall_year"] = long["fall_year"].astype(int)
    long = long[long["fall_year"] >= 2016]
    long["college"] = long["college_raw"].str.split("/").str[0].str.strip()
    long = long[long["level"].isin(["Undergraduate", "Graduate", "Professional"])]
    enrollment = long.groupby(["college", "fall_year", "level"], as_index=False)["headcount"].sum()
    enrollment["headcount"] = enrollment["headcount"].astype(int)
    assert not enrollment.duplicated(["college", "fall_year", "level"]).any()
    assert (enrollment["headcount"] >= 0).all()
    enrollment.to_csv(CLEAN / "dair_enrollment.csv", index=False)
    print(enrollment[enrollment["college"].str.startswith("Gies")].tail(8).to_string(index=False))
else:
    print("MISSING RAW FILE: data/raw/dair/dair_enrollment_historical.csv, so dair_enrollment.csv was not built")

blank enrollment cells:

 81 of 2926
                 college  fall_year         level  headcount
Gies College of Business       2023      Graduate       5790
Gies College of Business       2023 Undergraduate       3112
Gies College of Business       2024      Graduate       6132
Gies College of Business       2024 Undergraduate       3197
Gies College of Business       2025      Graduate       6311
Gies College of Business       2025 Undergraduate       3329
Gies College of Business       2026      Graduate       6443
Gies College of Business       2026 Undergraduate       3652


## IPEDS peers
One row per institution per year, for 2019-2023. The institution set is fixed from the 2023 directory: public universities with Carnegie 2021 Basic = 15 (doctoral, very high research). Revised `_rv` files are used wherever NCES has published them. Graduation rate is completers within 150% of normal time over the adjusted cohort (GRTYPE 3 / GRTYPE 2), which is the definition of "Graduation rate, total cohort".

In [6]:
def read(stem):
    revised = UNZ / f"{stem}_rv.csv"
    path = revised if revised.exists() else next(p for p in UNZ.iterdir() if p.name.lower() == f"{stem}.csv")
    frame = pd.read_csv(path, encoding="latin1", low_memory=False)
    frame.columns = frame.columns.str.strip().str.replace("ï»¿", "")
    return frame, path.name

hd, hd_name = read("hd2023")
base = hd[(hd["C21BASIC"] == 15) & (hd["CONTROL"] == 1)][["UNITID", "INSTNM", "STABBR"]]
base.columns = ["unitid", "institution", "state"]

# 16 of the 18 Big Ten members are public. Northwestern and USC are private, so they fall outside this set.
BIG_TEN = {145637: "Illinois", 151351: "Indiana", 153658: "Iowa", 163286: "Maryland", 170976: "Michigan",
           171100: "Michigan State", 174066: "Minnesota", 181464: "Nebraska", 204796: "Ohio State", 209551: "Oregon",
           214777: "Penn State", 243780: "Purdue", 186380: "Rutgers", 110662: "UCLA", 236948: "Washington", 240444: "Wisconsin"}
print(base[base["unitid"].isin(BIG_TEN)]["institution"].to_string(index=False))
assert base["unitid"].isin(BIG_TEN).sum() == 16

rows, files = [], []
for year in range(2019, 2024):
    ef_d, n1 = read(f"ef{year}d"); ef_a, n2 = read(f"ef{year}a"); gr, n3 = read(f"gr{year}")
    files.append({"year": year, "files": ", ".join([n1, n2, n3])})
    total = ef_a[ef_a["EFALEVEL"] == 1].set_index("UNITID")["EFTOTLT"]
    cohort = gr[gr["GRTYPE"] == 2].set_index("UNITID")["GRTOTLT"]
    done = gr[gr["GRTYPE"] == 3].set_index("UNITID")["GRTOTLT"]
    frame = base.copy()
    frame["year"] = year
    frame["retention_rate"] = frame["unitid"].map(ef_d.set_index("UNITID")["RET_PCF"])
    frame["grad_cohort"] = frame["unitid"].map(cohort)
    frame["grad_completers"] = frame["unitid"].map(done)
    frame["grad_rate_6yr"] = (100 * frame["grad_completers"] / frame["grad_cohort"]).round(1)
    frame["enrollment"] = frame["unitid"].map(total)
    rows.append(frame)
peers = pd.concat(rows, ignore_index=True)
print(pd.DataFrame(files).to_string(index=False))

blank = peers[["retention_rate", "grad_rate_6yr", "enrollment"]].isna().sum()
print("blank IPEDS cells:", blank.to_dict())
peers = peers.dropna(subset=["retention_rate", "grad_rate_6yr", "enrollment"]).copy()
peers["enrollment"] = peers["enrollment"].astype(int)
peers["big_ten"] = peers["unitid"].isin(BIG_TEN).map({True: "Big Ten", False: "Other public R1"})
peers["is_uiuc"] = (peers["unitid"] == 145637).map({True: "UIUC", False: "Peer"})
peers = peers.drop(columns=["grad_cohort", "grad_completers"])

assert peers["retention_rate"].between(0, 100).all() and peers["grad_rate_6yr"].between(0, 100).all()
assert not peers.duplicated(["unitid", "year"]).any()
assert peers["year"].between(2019, 2023).all()
peers.to_csv(CLEAN / "ipeds_peers.csv", index=False)
print(peers.groupby("year").size().to_string())
print(peers[peers["unitid"] == 145637].to_string(index=False))

     University of California-Los Angeles
  University of Illinois Urbana-Champaign
           Indiana University-Bloomington
                       University of Iowa
      University of Maryland-College Park
         University of Michigan-Ann Arbor
                Michigan State University
      University of Minnesota-Twin Cities
           University of Nebraska-Lincoln
         Rutgers University-New Brunswick
        Ohio State University-Main Campus
                     University of Oregon
Pennsylvania State University-Main Campus
  University of Washington-Seattle Campus
          University of Wisconsin-Madison
            Purdue University-Main Campus

 year                                         files
 2019 ef2019d_rv.csv, ef2019a_rv.csv, gr2019_rv.csv
 2020 ef2020d_rv.csv, ef2020a_rv.csv, gr2020_rv.csv
 2021 ef2021d_rv.csv, ef2021a_rv.csv, gr2021_rv.csv
 2022 ef2022d_rv.csv, ef2022a_rv.csv, gr2022_rv.csv
 2023          ef2023d.csv, ef2023a.csv, gr2023.csv
blank IPEDS cells: {'retention_rate': 7, 'grad_rate_6yr': 7, 'enrollment': 2}
year
2019    106
2020    105
2021    105
2022    106
2023    106
 unitid                             institution state  year  retention_rate  grad_rate_6yr  enrollment big_ten is_uiuc
 145637 University of Illinois Urbana-Champaign    IL  2019            93.0           85.1       51605 Big Ten    UIUC
 145637 University of Illinois Urbana-Champaign    IL  2020            94.0           86.2       52679 Big Ten    UIUC
 145637 University of Illinois Urbana-Champaign    IL  2021            93.0           84.9       56607 Big Ten    UIUC
 145637 University of Illinois Urbana-Champaign    IL  2022          